In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay



In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

def main():
    print("--- 🚀 Mini-Project: Ensemble Learning (Random Forest) ---")

    # ==========================================
    # 1. THE SETUP (Hardware & Determinism)
    # ==========================================
    RANDOM_STATE = 42
    N_JOBS = -1 # Use ALL CPU cores
    
    # Load Data (Simulating a CSV load)
    data = load_breast_cancer()
    X = data.data
    y = data.target
    feature_names = data.feature_names
    
    # Split: 80% Train/Tune, 20% Final Validation
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    
    # Strategy: Stratified K-Fold (Crucial for medical data to keep class ratios)
    cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    # ==========================================
    # 2. THE BASELINE (Sanity Check)
    # ==========================================
    print("\n[1/5] Establishing Baseline...")
    dummy = DummyClassifier(strategy="most_frequent")
    dummy.fit(X_train, y_train)
    baseline_score = dummy.score(X_test, y_test)
    print(f"   Baseline Accuracy (Blind Guessing): {baseline_score * 100:.2f}%")
    # If our fancy model doesn't beat this, we quit.

    # ==========================================
    # 3. THE TUNING (Optimization)
    # ==========================================
    print("\n[2/5] Tuning Random Forest...")
    
    # NICHE: 'class_weight="balanced"' 
    # This forces the model to pay more attention to the minority class (Cancer)
    # if the dataset is imbalanced.
    rf = RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced")
    
    param_grid = {
        'n_estimators': [50, 100, 200],      # Size of the "Committee"
        'max_depth': [None, 10, 20],         # Prevent individual trees from overfitting
        'max_features': ['sqrt', 'log2'],    # The "Feature Randomness" niche
        'min_samples_leaf': [1, 2, 4]        # Smooth out noise
    }
    
    grid = GridSearchCV(
        rf, 
        param_grid, 
        cv=cv_strategy, 
        scoring='recall', # NICHE: Optimize for RECALL (Catching all cancer cases)
        n_jobs=N_JOBS,
        verbose=1
    )
    
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    print(f"   Best Parameters: {grid.best_params_}")
    print(f"   Best Cross-Validation Recall: {grid.best_score_ * 100:.2f}%")

    # ==========================================
    # 4. THE DIAGNOSTICS (Health Check)
    # ==========================================
    print("\n[3/5] Diagnostics (Test Set)...")
    y_pred = best_model.predict(X_test)
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    
    print("\n   --- Classification Report ---")
    print(classification_report(y_test, y_pred, target_names=data.target_names))
    
    # NICHE CHECK: Look for False Negatives (Bottom Left of Matrix)
    # In Medicine, False Negative = Death. We want this to be 0.
    false_negatives = cm[1][0] 
    print(f"   CRITICAL CHECK: False Negatives count: {false_negatives}")
    if false_negatives > 0:
        print("   ⚠️ WARNING: Model missed some cancer cases. Adjustment needed.")
    else:
        print("   ✅ SUCCESS: No cancer cases missed.")

    # ==========================================
    # 5. THE INTERPRETATION (The "Why")
    # ==========================================
    print("\n[4/5] Interpretation (Feature Importance)...")
    
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1] # Sort descending
    
    print("   Top 5 Indicators:")
    for i in range(5):
        idx = indices[i]
        print(f"   {i+1}. {feature_names[idx]} ({importances[idx]:.4f})")
        
    # ==========================================
    # 6. THE SERIALIZATION (Deployment)
    # ==========================================
    print("\n[5/5] Saving Package...")
    
    package = {
        'model': best_model,
        'features': feature_names,
        'metadata': {
            'threshold': 0.5,
            'description': 'Breast Cancer Detection RF',
            'accuracy': best_model.score(X_test, y_test)
        }
    }
    
    joblib.dump(package, 'breast_cancer_rf_model.pkl')
    print("   ✅ Model saved to 'breast_cancer_rf_model.pkl'")

In [9]:
if __name__ == "__main__":
    main()

--- 🚀 Mini-Project: Ensemble Learning (Random Forest) ---

[1/5] Establishing Baseline...
   Baseline Accuracy (Blind Guessing): 63.16%

[2/5] Tuning Random Forest...
Fitting 5 folds for each of 54 candidates, totalling 270 fits
   Best Parameters: {'max_depth': 10, 'max_features': 'log2', 'min_samples_leaf': 1, 'n_estimators': 50}
   Best Cross-Validation Recall: 98.25%

[3/5] Diagnostics (Test Set)...

   --- Classification Report ---
              precision    recall  f1-score   support

   malignant       0.93      0.93      0.93        42
      benign       0.96      0.96      0.96        72

    accuracy                           0.95       114
   macro avg       0.94      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114

   CRITICAL CHECK: False Negatives count: 3
   ⚠️ WARNING: Model missed some cancer cases. Adjustment needed.

[4/5] Interpretation (Feature Importance)...
   Top 5 Indicators:
   1. worst radius (0.1125)
   2. mean perimeter (0.110